# 2. 2Wiki reference example: text-to-KG reasoning

This walkthrough follows exactly one fixed raw record:

- file: `data/raw/2WikiMultihopQA/data/train-00000-of-00002.parquet`
- id: `13f5ad2c088c11ebbd6fac1f6bf848b6`
- question: **Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?**

No other 2Wiki record is selected or processed.

In [1]:
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data.build_subgraph_training_data import (
    beam_connected_subgraphs,
    materialize_retrieval_rows,
    set_scores,
)
from data_processing.build_2wiki_subgraph_dataset import (
    align_gold_evidence_to_candidate_kg,
    evidence_triple_units,
    flatten_context,
    normalize_2wiki_record,
    select_kg_pool,
)
from data_processing.text_kg_constructor import (
    KGConstructionConfig,
    LLMKGConstructor,
    construct_text_kg,
    construct_wiki_bridge_triples,
    dedupe_triples,
)

## What the raw 2Wiki dataset contains

The Parquet record has seven fields:

| Raw field | SAGE-QA use for this example |
|---|---|
| `question` | **Inference input:** query for KG construction, retrieval, and answering |
| `context` | **Inference input:** Wikipedia text from which the candidate KG is constructed |
| `evidences` | **Gold supervision/evaluation only:** target triple reasoning path |
| `answer` | **Gold answer supervision/evaluation only** |
| `id` | Stable example identifier |
| `type` | Metadata and diagnostics |
| `supporting_facts` | Not used by this graph-level SAGE-QA pipeline |

In [2]:
raw_path = REPO_ROOT / "data/raw/2WikiMultihopQA/data/train-00000-of-00002.parquet"
REFERENCE_ID = "13f5ad2c088c11ebbd6fac1f6bf848b6"

raw_frame = pd.read_parquet(raw_path, filters=[("id", "==", REFERENCE_ID)])
assert len(raw_frame) == 1
raw = raw_frame.iloc[0].to_dict()
assert raw["id"] == REFERENCE_ID

example = normalize_2wiki_record(raw)

print("Raw file:", raw_path.relative_to(REPO_ROOT))
print("Reference id:", REFERENCE_ID)
print("Raw fields:", list(raw))
print("Question type:", raw["type"])
print("Question:", example["question"])
print("Context pages:", len(example["context"]))
print("Raw supporting_facts present:", "supporting_facts" in raw)
print("supporting_facts retained by SAGE-QA:", "supporting_facts" in example)

Raw file: data\raw\2WikiMultihopQA\data\train-00000-of-00002.parquet
Reference id: 13f5ad2c088c11ebbd6fac1f6bf848b6
Raw fields: ['id', 'question', 'answer', 'type', 'evidences', 'supporting_facts', 'context']
Question type: bridge_comparison
Question: Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Context pages: 10
Raw supporting_facts present: True
supporting_facts retained by SAGE-QA: False


## Separate inference inputs from gold labels

In [3]:
# Available to SAGE-QA when constructing and reasoning over the KG.
# Raw context is grouped by Wikipedia page. flatten_context creates one
# record per sentence: {title, sent_idx, sentence}. It adds no information.

question = example["question"]
sentence_records = flatten_context(example)

# Used only to train or evaluate the prediction.
gold_answer = example["answer"]
raw_gold_evidence_units = evidence_triple_units(example["evidences"])

print("Inference question:", question)
print("Context sentences available to text-to-KG:", len(sentence_records))
print("Gold answer:", gold_answer)
print("Gold evidence path (KG::<subject>::<predicate>::<object>):")
print(*raw_gold_evidence_units, sep="\n")

Inference question: Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Context sentences available to text-to-KG: 16
Gold answer: no
Gold evidence path (KG::<subject>::<predicate>::<object>):
KG::Move (1970 film)::directed_by::Stuart Rosenberg
KG::Méditerranée (1963 film)::directed_by::Jean-Daniel Pollet
KG::Stuart Rosenberg::country_of_citizenship::American
KG::Jean-Daniel Pollet::country_of_citizenship::French


## Construct the candidate KG from `question + context`

### One-sentence construction illustration

Before processing all 16 context sentences, follow one fixed sentence through every construction case. This sentence is an inference input from the `Move (1970 film)` context page; it is not selected from the gold evidence annotations.

In [4]:
illustration_record = next(
    record
    for record in sentence_records
    if record["title"] == "Move (1970 film)" and record["sent_idx"] == 0
)

print("Title:", illustration_record["title"])
print("Sentence index:", illustration_record["sent_idx"])
print("Sentence:", illustration_record["sentence"])

Title: Move (1970 film)
Sentence index: 0
Sentence: Move is a 1970 American comedy film starring Elliott Gould, Paula Prentiss and Geneviève Waïte, and directed by Stuart Rosenberg.


The sentence says that *Move* was **directed by Stuart Rosenberg**. Here is what each construction method does with that same input:

| Method | Operation on this sentence | Result for the illustration |
|---|---|---|
| `deterministic` | Notice that the sentence contains another context-page title, `Stuart Rosenberg` | `KG::Move (1970 film)::mentions_page::Stuart Rosenberg` |
| `llm` | Ask the model to identify the stated semantic relation | A plausible extraction is `KG::Move (1970 film)::directed_by::Stuart Rosenberg` |
| `llm_with_title_bridges` | Keep both the semantic LLM triple and deterministic title bridge | Both triples above, after deduplication |

The LLM triple shown here is an **illustrative possible extraction**.

### Executable setup for the one-sentence cases

Set `RUN_ONE_SENTENCE_LLM = True` to execute the two cases that call the configured LLM. With `False`, every cell remains executable without credentials and the LLM-dependent result is an empty list.

The semantic LLM receives exactly one sentence. The deterministic constructor also needs the list of context-page titles so it can recognize `Stuart Rosenberg` as a page mention; the extra records below provide only that title inventory and contain no additional sentence text.

In [5]:
RUN_ONE_SENTENCE_LLM = True
ONE_SENTENCE_LLM_MODEL = "openai:gpt-4.1-mini"

one_sentence_records = [illustration_record]
context_page_titles = list(
    dict.fromkeys(record["title"] for record in sentence_records)
)
one_sentence_with_title_inventory = [
    illustration_record,
    *[
        {"title": title, "sent_idx": -1, "sentence": ""}
        for title in context_page_titles
        if title != illustration_record["title"]
    ],
]

print("Semantic sentence inputs:", len(one_sentence_records))
print(
    "Context-page titles available to deterministic matching:", len(context_page_titles)
)

Semantic sentence inputs: 1
Context-page titles available to deterministic matching: 10


#### Case 1 — `deterministic`

Execute the title co-mention constructor. It sees the phrase `Stuart Rosenberg`, recognizes it as another context-page title, and emits a structural bridge rather than the semantic `directed_by` relation.

In [6]:
one_sentence_title_bridges = construct_wiki_bridge_triples(
    sentence_records=one_sentence_with_title_inventory,
    max_triples=16,
)
print(*evidence_triple_units(one_sentence_title_bridges), sep="\n")

KG::Move (1970 film)::mentions_page::Stuart Rosenberg


#### Case 2 — `llm`

Execute semantic extraction on exactly the selected sentence. When the flag is `False`, the cell explicitly skips the API call. Set it to `True` in the setup cell and rerun from there to obtain the model's actual triples.

In [7]:
one_sentence_llm_config = KGConstructionConfig(
    backend="llm",
    model=ONE_SENTENCE_LLM_MODEL,
    max_triples=16,
    max_context_sentences=1,
)

if RUN_ONE_SENTENCE_LLM:
    one_sentence_llm_constructor = LLMKGConstructor(one_sentence_llm_config)
    one_sentence_llm_triples = one_sentence_llm_constructor.construct(
        question=question,
        sentence_records=one_sentence_records,
    )
    print(*evidence_triple_units(one_sentence_llm_triples), sep="\n")
else:
    one_sentence_llm_triples = []
    print("LLM call skipped. Set RUN_ONE_SENTENCE_LLM = True to execute it.")

KG::Move (1970 film)::directed_by::Stuart Rosenberg
KG::Move (1970 film)::country_of::United States


#### Case 3 — `llm_with_title_bridges`

Execute the union performed by the combined backend. If the LLM call was skipped, the output contains only deterministic bridges. If it ran successfully, the output contains semantic triples plus bridges, with duplicates removed.

In [8]:
one_sentence_combined_triples = dedupe_triples(
    [*one_sentence_llm_triples, *one_sentence_title_bridges]
)
print(*evidence_triple_units(one_sentence_combined_triples), sep="\n")

KG::Move (1970 film)::directed_by::Stuart Rosenberg
KG::Move (1970 film)::country_of::United States
KG::Move (1970 film)::mentions_page::Stuart Rosenberg


### Apply one backend to the full `question + context`

```text
question + context
    │ text-to-KG bridge
    ▼
independent candidate triples
    │ candidate composition
    ▼
candidate reasoning paths
    │ GNN ranking
    ▼
predicted reasoning path

Gold evidences ──► training supervision and evaluation only
```

#### Available construction methods for the full-context run

| `KG_BACKEND` method | What it constructs | `construction_method` when successful | Requires an LLM call? |
|---|---|---|---:|
| `deterministic` | `mentions_page` triples from context-page title co-mentions | `title_bridge_fallback` | No |
| `llm` | Semantic triples extracted from question-relevant context sentences | `llm` | Yes |
| `llm_with_title_bridges` | The union of semantic LLM triples and deterministic title bridges | `llm_plus_title_bridges` | Yes |

If a selected method returns no valid triples, `construction_method` is recorded as `none`. This is an empty-result outcome, not a fourth construction method.

In [9]:
KG_BACKEND = (
    "llm_with_title_bridges"  # options: deterministic, llm, llm_with_title_bridges
)
SUPPORTED_KG_BACKENDS = {
    "deterministic",
    "llm",
    "llm_with_title_bridges",
}
assert KG_BACKEND in SUPPORTED_KG_BACKENDS

uses_llm = KG_BACKEND in {"llm", "llm_with_title_bridges"}
config = KGConstructionConfig(
    backend=KG_BACKEND,
    model="openai:gpt-4.1-mini",
    max_triples=64,
    max_context_sentences=30,
)
constructor = LLMKGConstructor(config) if uses_llm else None

kg_triples, construction_method = construct_text_kg(
    sentence_records=sentence_records,
    question=question,
    config=config,
    llm_constructor=constructor,
)
candidate_kg_units = select_kg_pool(
    question=question,
    kg_units=evidence_triple_units(kg_triples),
    max_candidate_units=30,
)

print("Requested backend:", KG_BACKEND)
print("Construction method used:", construction_method)
print("Candidate KG facts:", len(candidate_kg_units))
print(*candidate_kg_units, sep="\n")

Requested backend: llm_with_title_bridges
Construction method used: llm_plus_title_bridges
Candidate KG facts: 8
KG::Move (1970 film)::directed_by::Stuart Rosenberg
KG::Stuart Rosenberg::occupation::film director
KG::Stuart Rosenberg::country_of::American
KG::Méditerranée (1963 film)::directed_by::Jean-Daniel Pollet
KG::Jean-Daniel Pollet::occupation::film director
KG::Jean-Daniel Pollet::country_of::French
KG::Méditerranée (1963 film)::mentions_page::Jean-Daniel Pollet
KG::Move (1970 film)::mentions_page::Stuart Rosenberg


#### Inspect all context sentences

This optional inspection cell prints all 16 flattened sentence records as `page title: sentence index`, followed by the sentence text. It does not construct triples or change the candidate KG.

In [10]:
for sentence_record in sentence_records:
    print(f"{sentence_record['title']}: {sentence_record['sent_idx']}")
    print(f"{sentence_record['sentence']}\n")

Stuart Rosenberg: 0
Stuart Rosenberg (August 11, 1927 – March 15, 2007) was an American film and television director whose motion pictures include "Cool Hand Luke" (1967), "Voyage of the Damned" (1976), "The Amityville Horror" (1979), and "The Pope of Greenwich Village" (1984).

Stuart Rosenberg: 1
He was noted for his work with actor Paul Newman.

Méditerranée (1963 film): 0
Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff.

Méditerranée (1963 film): 1
It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel.

Méditerranée (1963 film): 2
The 45 minute film is cited as one of Pollet's most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard's "Contempt", released later the same year.

Méditerranée (1963 film): 3
Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also fe

### Diagnose KG extraction against the gold triples

Only after construction is complete do we compare the candidate KG with the annotated `evidences`. This calculates a gold-derived diagnostic score for training/evaluation without inserting missing gold facts into the candidate KG.

$\text{gold\_kg\_coverage} = |G \cap K| / |G|$, where $G$ is the set of gold evidence triples and $K$ is the constructed candidate KG. It is analogous to FamilyOWL's `gold_context_coverage` and is not a GNN input.

In [11]:
aligned_gold_units, gold_kg_coverage = align_gold_evidence_to_candidate_kg(
    raw_gold_evidence_units,
    candidate_kg_units,
)
print("Raw gold evidence triples:", len(raw_gold_evidence_units))
print("Gold target triples after spelling alignment:", len(aligned_gold_units))
print(
    "Gold triples matched in candidate KG:",
    round(gold_kg_coverage * len(raw_gold_evidence_units)),
)
print("Gold KG extraction coverage (diagnostic only):", gold_kg_coverage)
print("Missing gold facts remain missing; they are not inserted into candidates.")

Raw gold evidence triples: 4
Gold target triples after spelling alignment: 4
Gold triples matched in candidate KG: 2
Gold KG extraction coverage (diagnostic only): 0.5
Missing gold facts remain missing; they are not inserted into candidates.


## Compose candidate reasoning paths

The `candidate_kg_units` above are independent triples. From this point onward, 2Wiki uses the same `beam_connected_subgraphs` composer as FamilyOWL. It combines the units into bounded, connected candidate reasoning paths for the GNN to rank. Candidate generation uses only the question and constructed KG; `sparql_query=""` because 2Wiki has no SPARQL query.

During this search, an internal heuristic score keeps combinations that are structurally connected and relevant to the question. Its only purpose is to control the beam and avoid enumerating every possible combination. It is not the GNN score and does not use the gold evidence.

After candidates exist, the same `set_scores` function used for Family compares each one with the gold evidence triples. This is post-hoc oracle scoring for training and evaluation; it does not influence text-to-KG construction or candidate generation.

In [12]:
# This is the same composer used in the Family walkthrough.
# Gold evidence triples are not provided to this function.
candidate_subgraphs = beam_connected_subgraphs(
    candidate_units=candidate_kg_units,
    question=question,
    sparql_query="",
    min_subgraph_size=1,
    max_subgraph_size=4,
    beam_width=96,
    max_candidate_subgraphs=256,
)
candidates = [list(candidate) for candidate in sorted(candidate_subgraphs)]
# Post-hoc oracle scoring for supervision/evaluation only.
oracle_scored_candidates = sorted(
    (
        (set_scores(candidate, [aligned_gold_units]), candidate)
        for candidate in candidates
    ),
    key=lambda item: item[0]["best_set_f1_to_gold"],
    reverse=True,
)

print("Generated candidate paths:", len(candidates))
if oracle_scored_candidates:
    oracle_metrics, oracle_best_candidate = oracle_scored_candidates[0]
    print(
        "Supervision/evaluation metrics:",
        oracle_metrics,
    )
    print("Oracle-best candidate for this reference example:")
    print(*oracle_best_candidate, sep="\n")

Generated candidate paths: 112
Supervision/evaluation metrics: {'best_jaccard_to_gold': 0.5, 'best_set_precision_to_gold': 1.0, 'best_set_recall_to_gold': 0.5, 'best_set_f1_to_gold': 0.6666666666666666, 'exact_match_any_gold': False, 'contains_any_gold_explanation': False, 'contained_in_any_gold_explanation': True, 'best_matching_gold_explanation': ['KG::Move (1970 film)::directed_by::Stuart Rosenberg', 'KG::Méditerranée (1963 film)::directed_by::Jean-Daniel Pollet', 'KG::Stuart Rosenberg::country_of_citizenship::American', 'KG::Jean-Daniel Pollet::country_of_citizenship::French'], 'best_matching_gold_index': 0}
Oracle-best candidate for this reference example:
KG::Move (1970 film)::directed_by::Stuart Rosenberg
KG::Méditerranée (1963 film)::directed_by::Jean-Daniel Pollet


## Materialize the GNN retrieval rows

The GNN scores candidate paths, not the independent `candidate_kg_units` directly. The shared `materialize_retrieval_rows` function creates one row for each candidate path. This is the same function used in the Family walkthrough; 2Wiki does not call a second dataset-specific row builder here.

- `subgraph_units` and `subgraph_node_ids` identify the candidate presented to the model.
- `label` says whether the candidate contains the complete aligned gold evidence path.
- `best_set_f1_to_gold` measures triple-set overlap with that aligned gold path.

For candidate set $C$ and gold set $G$, precision is $|C \cap G|/|C|$, recall is $|C \cap G|/|G|$, and Gold F1 is their harmonic mean. During training, this overlap contributes to the GNN ranking target; it is not an input feature. At inference time, the gold fields are unavailable.

In [13]:
reference_example_id = f"2WikiMultiHopQA__reference__{REFERENCE_ID}"
retrieval_rows = materialize_retrieval_rows(
    candidate_units=candidate_kg_units,
    candidate_subgraphs=candidate_subgraphs,
    gold_explanations=[aligned_gold_units],
    base_row={
        "example_id": reference_example_id,
        "dataset": "2WikiMultiHopQA",
        "question": question,
        "sparql_query": "",
        "answer": gold_answer,
        "answer_type": "BIN",
        "evidence_unit_type": "kg_triple",
        "kg_construction_method": construction_method,
        "candidate_kg_units": candidate_kg_units,
        "gold_explanations": [aligned_gold_units],
        "gold_units": aligned_gold_units,
        "raw_gold_evidence_units": raw_gold_evidence_units,
        "gold_kg_coverage": gold_kg_coverage,
    },
)

assert len({row["example_id"] for row in retrieval_rows}) == 1
print("Rows for this one question:", len(retrieval_rows))
print("Positive training rows:", sum(row["label"] for row in retrieval_rows))
print("Example candidate row:")
print(
    {
        key: retrieval_rows[0][key]
        for key in (
            "subgraph_node_ids",
            "subgraph_units",
            "label",
            "best_set_f1_to_gold",
            "gold_kg_coverage",
        )
    }
)

Rows for this one question: 112
Positive training rows: 0
Example candidate row:
{'subgraph_node_ids': [0, 3], 'subgraph_units': ['KG::Move (1970 film)::directed_by::Stuart Rosenberg', 'KG::Méditerranée (1963 film)::directed_by::Jean-Daniel Pollet'], 'label': 0, 'best_set_f1_to_gold': 0.6666666666666666, 'gold_kg_coverage': 0.5}


## Prepare the shared local graph

`prepare_examples` reconstructs the local KG node universe from the retrieval rows and maps each candidate path to node IDs. The GNN encodes the question and graph nodes, performs message passing over the shared graph, and pools the node subset belonging to each candidate.

The graph size depends on the selected construction method. With the currently selected `llm_with_title_bridges` method, the saved construction contains six semantic LLM triples and two deterministic `mentions_page` bridges. `prepare_examples` uses their union as the local node universe; it does not use the gold triples to add nodes.

In [14]:
import torch

from models.gnn_subgraph_retriever import GNNSubgraphRetriever
from training.train_gnn_subgraph_retriever import (
    encode_example_graph,
    prepare_examples,
    score_candidate_rows,
)
from utils.tokenizer import load_tokenizer

prepared_examples = prepare_examples(retrieval_rows)
assert len(prepared_examples) == 1
gnn_example = prepared_examples[0]

print("Local GNN nodes:", len(gnn_example["candidate_axioms"]))
print("Candidate paths to score:", len(gnn_example["candidate_rows"]))
print("Has distinct training targets:", gnn_example["has_rankable_pairs"])
print(
    "First candidate node IDs:", gnn_example["candidate_rows"][0]["subgraph_node_ids"]
)

c:\Users\julie\github\PhD\SAGE-QA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Local GNN nodes: 8
Candidate paths to score: 112
Has distinct training targets: True
First candidate node IDs: [0, 6, 1, 4]


## Run GNN retrieval

This cell loads the local 2Wiki checkpoint, constructs the message-passing graph, and assigns a learned GNN probability to each candidate path. At inference time, candidates are ranked only by this probability.

Gold F1 is printed beside the result only as a post-hoc evaluation diagnostic. It is not a model prediction and is not passed to the GNN during inference. With `llm_with_title_bridges`, the GNN ranks paths composed from both semantic LLM triples and deterministic title bridges. Rerun this section whenever the constructed KG changes so that its rankings refer to the current candidates.

The existing checkpoint may predate the current inference-safe 2Wiki KG builder. Rebuild the dataset and retrain the GNN before treating its scores as manuscript results.

In [15]:
CHECKPOINT = REPO_ROOT / "checkpoints/gnn_subgraph_ranker_2wiki_full/best_model.pt"
RUN_GNN = CHECKPOINT.exists()

if not RUN_GNN:
    print("Checkpoint not found; expected:", CHECKPOINT)
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
    model_name = checkpoint["model_name"]
    tokenizer = load_tokenizer(model_name)
    model = GNNSubgraphRetriever(
        model_name=model_name,
        node_symbolic_dim=checkpoint.get("node_symbolic_dim", 8),
        subgraph_symbolic_dim=checkpoint.get("subgraph_symbolic_dim", 8),
        gnn_hidden_dim=checkpoint.get("gnn_hidden_dim", 128),
        gnn_layers=checkpoint.get("gnn_layers", 2),
        classifier_hidden_dim=checkpoint.get("classifier_hidden_dim", 128),
        freeze_encoder=checkpoint.get("freeze_encoder", False),
        architecture_version=checkpoint.get("architecture_version", 1),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    with torch.no_grad():
        encoded_graph = encode_example_graph(
            model=model,
            tokenizer=tokenizer,
            example=gnn_example,
            device=device,
            max_length=128,
        )
        output = score_candidate_rows(
            model=model,
            encoded_graph=encoded_graph,
            candidate_rows=gnn_example["candidate_rows"],
            device=device,
        )

    ranked_rows = [
        {**row, "gnn_score": float(probability)}
        for row, probability in zip(
            gnn_example["candidate_rows"], output["probs"].cpu().tolist()
        )
    ]
    ranked_rows.sort(key=lambda row: row["gnn_score"], reverse=True)

    print("Device:", device)
    print("Encoder:", model_name)
    print("Directed edges including self-loops:", encoded_graph["edge_index"].shape[1])
    print("GNN retrieval results:")
    for rank, row in enumerate(ranked_rows, start=1):
        print(f"\nRank {rank} | score={row['gnn_score']:.6f}")
        print(*row["subgraph_units"], sep="\n")
        print("Gold F1 (evaluation only):", row["best_set_f1_to_gold"])

Device: cpu
Encoder: google/bert_uncased_L-2_H-128_A-2
Directed edges including self-loops: 34
GNN retrieval results:

Rank 1 | score=0.329485
KG::Move (1970 film)::directed_by::Stuart Rosenberg
KG::Stuart Rosenberg::occupation::film director
KG::Méditerranée (1963 film)::directed_by::Jean-Daniel Pollet
KG::Jean-Daniel Pollet::occupation::film director
Gold F1 (evaluation only): 0.5

Rank 2 | score=0.319668
KG::Move (1970 film)::directed_by::Stuart Rosenberg
KG::Stuart Rosenberg::occupation::film director
KG::Méditerranée (1963 film)::directed_by::Jean-Daniel Pollet
KG::Méditerranée (1963 film)::mentions_page::Jean-Daniel Pollet
Gold F1 (evaluation only): 0.5

Rank 3 | score=0.302854
KG::Move (1970 film)::directed_by::Stuart Rosenberg
KG::Méditerranée (1963 film)::directed_by::Jean-Daniel Pollet
KG::Jean-Daniel Pollet::occupation::film director
KG::Méditerranée (1963 film)::mentions_page::Jean-Daniel Pollet
Gold F1 (evaluation only): 0.5

Rank 4 | score=0.293348
KG::Move (1970 film)::d

## Reference-example conclusion

For this one 2Wiki example, SAGE-QA consumes text plus a question, constructs a local KG, composes candidate triple paths, and sends those paths to the GNN retriever. The gold `evidences` remain triple-level supervision and evaluation labels throughout.

`gold_kg_coverage` separates the two failure stages: **text-to-KG extraction** determines whether the required reasoning triples exist in the candidate KG, while **GNN retrieval** determines whether an existing correct path is ranked highly.

In the currently saved eight-triple hybrid construction, the two `directed_by` triples match the gold evidence, while the country facts use different predicates and values (`country_of → United States/France` rather than `country_of_citizenship → American/French`). A fresh coverage calculation therefore yields `0.5`. The complete gold path is still unavailable until the text-to-KG bridge recovers or normalizes the two country triples.